# CUDA-zkML / PCANI — Gate F replication (2026)

Runs `scripts/run_gate_f.sh`: clean build for the detected GPU, full regression, then protocol-v2 on

* **digits** — the completed T4 experiment, unchanged (GPU replication),
* **mnist_mlp** — larger benchmark, same model family,
* **mnist_conv** — new model family (convolution lowered to the exact LINEAR circuit),

plus a matched-workload EZKL comparison on the same exact-integer models. Everything is fail-closed; the result is one ZIP.

**Pick a GPU other than the T4 used in the first run** (e.g. L4 or A100) so the digits run counts as a replication on another GPU.

In [ ]:
# 1) Confirm NVIDIA/CUDA runtime
import subprocess
print(subprocess.check_output(['nvidia-smi'], text=True))
print(subprocess.check_output(['nvcc', '--version'], text=True))

In [ ]:
# 2) Upload the project ZIP
from google.colab import files
from pathlib import Path
import zipfile, shutil, os

uploaded = files.upload()
zip_names = [n for n in uploaded if n.lower().endswith('.zip')]
if len(zip_names) != 1:
    raise RuntimeError(f'Upload exactly one project ZIP; got: {zip_names}')
extract_root = Path('/content/pcani_src')
if extract_root.exists():
    shutil.rmtree(extract_root)
extract_root.mkdir(parents=True)
with zipfile.ZipFile(zip_names[0]) as zf:
    zf.extractall(extract_root)
repos = [p.parent for p in extract_root.rglob('CMakeLists.txt') if (p.parent / 'scripts' / 'run_gate_f.sh').exists()]
if len(repos) != 1:
    raise RuntimeError(f'Could not identify a unique repository root: {repos}')
REPO = repos[0]
os.chdir(REPO)
print('Repository:', REPO)

In [ ]:
# 3) System packages
!apt-get -qq update
!apt-get -qq install -y cmake build-essential zip > /dev/null
!python -m pip -q install --upgrade pip

In [ ]:
# 4) Run Gate F. Adjust WORKLOADS/REPEATS only if the session time limit requires it
#    (e.g. WORKLOADS='digits mnist_mlp' first, then 'mnist_conv' in a new session).
import os, subprocess
env = dict(os.environ, WORKLOADS='digits mnist_mlp mnist_conv', REPEATS='5', EZKL='1')
subprocess.run(['bash', 'scripts/run_gate_f.sh'], cwd=REPO, env=env, check=True)

In [ ]:
# 5) Show the measured summary. Do not edit these numbers manually.
import json
runs = sorted((REPO / 'results' / 'gate_f').iterdir())
latest = runs[-1]
print(json.dumps(json.loads((latest / 'gate_f_summary.json').read_text()), indent=2))

In [ ]:
# 6) Download the archive
from google.colab import files
zips = sorted(REPO.glob('GateF_*.zip'))
if not zips:
    raise FileNotFoundError('GateF_*.zip')
print('Result ZIP:', zips[-1], zips[-1].stat().st_size, 'bytes')
files.download(str(zips[-1]))